In [1]:
import math
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Literal

import numpy as np
import numpy.typing as npt
import pandas as pd
import plotly.graph_objects as go
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore")

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
DATA_DIR = Path("../data")
CHECKPOINT_DIR = Path("./checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)
DEVICE = (
    torch.device("cuda") if torch.cuda.is_available() else 
    torch.device("mps") if torch.backends.mps.is_available() else
    torch.device("cpu")
)
print(f"Device: {DEVICE}")

Device: mps


## Data Loading

In [4]:
class WindowDataset(Dataset):
    """Sliding-window dataset for direction model. Target = signed return."""
    def __init__(
        self,
        df: pd.DataFrame,
        feature_cols: list[str],
        seq_len: int = 20,
        target_col: str = "target",
    ):
        self.seq_len = seq_len
        self.samples: list[tuple[npt.NDArray[np.float32], np.float32]] = []

        for ticker, group in df.groupby("ticker"):
            group = group.sort_index()
            X = group[feature_cols].values.astype(np.float32)
            y = group[target_col].values.astype(np.float32)

            mask = np.isfinite(X).all(axis=1) & np.isfinite(y)
            X = X[mask]
            y = y[mask]

            for i in range(seq_len, len(group)):
                window = X[i - seq_len : i]
                target = y[i]
                self.samples.append((window, target))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.from_numpy(x), torch.tensor(y)


class VolatilityDataset(Dataset):
    """Sliding-window dataset for volatility model. Target = |return|."""
    def __init__(
        self,
        df: pd.DataFrame,
        feature_cols: list[str],
        seq_len: int = 20,
        target_col: str = "target",
    ):
        self.seq_len = seq_len
        self.samples: list[tuple[npt.NDArray[np.float32], np.float32]] = []

        for ticker, group in df.groupby("ticker"):
            group = group.sort_index()
            X = group[feature_cols].values.astype(np.float32)
            y = np.abs(group[target_col].values.astype(np.float32))  # ← absolute value

            mask = np.isfinite(X).all(axis=1) & np.isfinite(y)
            X = X[mask]
            y = y[mask]

            for i in range(seq_len, len(group)):
                window = X[i - seq_len : i]
                target = y[i]
                self.samples.append((window, target))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.from_numpy(x), torch.tensor(y)

In [5]:
def make_loaders(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    feature_cols: list[str],
    dataset_cls: type[Dataset],
    seq_len: int = 20,
    batch_size: int = 64,
) -> tuple[DataLoader, DataLoader]:
    kw = dict(feature_cols=feature_cols, seq_len=seq_len)
    train_ds = dataset_cls(train_df, **kw)
    val_ds   = dataset_cls(val_df,   **kw)

    loader_kw = dict(batch_size=batch_size, num_workers=0)
    return (
        DataLoader(train_ds, shuffle=True,  **loader_kw),
        DataLoader(val_ds,   shuffle=False, **loader_kw),
    )

## Direction Model (GRU + Attention)

Identical to your working single-head model. This is Grumbert — don't touch what works.

In [6]:
class SoftDirectionalHuberLoss(nn.Module):
    """Huber + differentiable directional penalty."""

    def __init__(self, delta=1.01, dir_weight=0.3, sharpness=10.0):
        super().__init__()
        self.huber = nn.HuberLoss(delta=delta, reduction="none")
        self.dir_weight = dir_weight
        self.sharpness = sharpness

    def forward(self, preds, targets):
        huber = self.huber(preds, targets).mean()
        agreement = torch.tanh(preds * self.sharpness) * torch.tanh(targets * self.sharpness)
        dir_loss = -agreement.mean()
        return huber + self.dir_weight * dir_loss

In [7]:
class AdditiveAttention(nn.Module):
    """Bahdanau-style attention over the time dimension."""

    def __init__(self, hidden_size: int):
        super().__init__()
        self.W = nn.Linear(hidden_size, hidden_size, bias=True)
        self.v = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, hidden_states: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        scores = self.v(torch.tanh(self.W(hidden_states)))  # (B, T, 1)
        weights = torch.softmax(scores, dim=1)               # (B, T, 1)
        context = (weights * hidden_states).sum(dim=1)        # (B, H)
        return context, weights.squeeze(-1)

In [8]:
class GRUWithAttention(nn.Module):
    """Direction model — your working single-head architecture."""
    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        num_layers: int = 1,
        linear_hidden: int = 64,
        dropout: float = 0.2,
    ):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.attention = AdditiveAttention(hidden_size)
        self.head = nn.Sequential(
            nn.Linear(hidden_size, linear_hidden),
            nn.GELU(),
            nn.LayerNorm(linear_hidden),
            nn.Dropout(dropout),
            nn.Linear(linear_hidden, 1),
        )

    def forward(self, x: torch.Tensor):
        hidden_states, _ = self.gru(x)
        context, attn = self.attention(hidden_states)
        out = self.head(context).squeeze(-1)
        return out, attn

## Volatility Model

Separate, smaller GRU that predicts |return| (magnitude only).
Volatility clusters — big moves follow big moves — so this is an
easier task than direction. Uses its own feature set focused on
recent volatility, volume, and range.

In [9]:
class VolatilityGRU(nn.Module):
    """
    Small GRU for predicting |return| (next-day absolute return).
    
    Simpler than the direction model because volatility clustering
    is a stronger, more persistent signal than directional prediction.
    Uses Softplus output to ensure positive predictions.
    """
    def __init__(
        self,
        input_size: int,
        hidden_size: int = 16,
        num_layers: int = 1,
        linear_hidden: int = 16,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size, linear_hidden),
            nn.GELU(),
            nn.LayerNorm(linear_hidden),
            nn.Dropout(dropout),
            nn.Linear(linear_hidden, 1),
            nn.Softplus(),  # Ensures positive output
        )

    def forward(self, x: torch.Tensor):
        hidden_states, _ = self.gru(x)
        last = hidden_states[:, -1, :]      # last timestep only (no attention needed)
        out = self.head(last).squeeze(-1)    # (B,)
        return out

## Training Functions

Generic training loop that works for both models. The direction model
returns `(pred, attn)`, the volatility model returns just `pred` —
we handle both.

In [10]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
    clip_grad: float = 1.0,
) -> float:
    model.train()
    total_loss = 0.0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x)
        preds = out[0] if isinstance(out, tuple) else out
        loss = criterion(preds, y)
        loss.backward()

        if clip_grad > 0:
            nn.utils.clip_grad_norm_(model.parameters(), clip_grad)

        optimizer.step()
        total_loss += loss.item() * len(y)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> dict:
    model.eval()
    all_preds, all_targets = [], []

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        preds = out[0] if isinstance(out, tuple) else out
        all_preds.append(preds.cpu())
        all_targets.append(y.cpu())

    preds = torch.cat(all_preds)
    targets = torch.cat(all_targets)

    loss = criterion(preds, targets).item()
    mae = (preds - targets).abs().mean().item()
    
    # Direction accuracy (meaningful for direction model, less so for vol model)
    dir_acc = (preds.sign() == targets.sign()).float().mean().item()

    return {"loss": loss, "mae": mae, "dir_acc": dir_acc}


def fit_model(
    model:        nn.Module,
    train_loader: DataLoader,
    val_loader:   DataLoader,
    criterion:    nn.Module,
    device:       torch.device,
    lr:           float = 1e-3,
    epochs:       int   = 1000,
    patience:     int   = 20,
    weight_decay: float = 1e-2,
    warmup_epochs: int  = 10,
    tag:          str   = "",
    verbose:      bool  = True,
) -> nn.Module:
    """
    Generic training loop for any model.
    
    Handles linear warmup → cosine decay, early stopping, grad clipping.
    Works for both direction (GRUWithAttention) and volatility (VolatilityGRU).
    """
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, epochs - warmup_epochs)
        return 0.5 * (1 + math.cos(math.pi * progress))
    
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    best_val_loss = float("inf")
    patience_ctr  = 0
    best_state    = None

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_metrics = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        if verbose:
            print(
                f"  {tag} Epoch {epoch:3d} | "
                f"train_loss={train_loss:.4f}  "
                f"val_loss={val_metrics['loss']:.4f}  "
                f"val_mae={val_metrics['mae']:.3f}  "
                f"dir_acc={val_metrics['dir_acc']:.3f}"
            )

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                if verbose:
                    print(f"  {tag} Early stop at epoch {epoch}.")
                break

    model.load_state_dict(best_state)
    return model

## Combined Prediction

The two models are independent — they don't share parameters.  
We just combine their outputs:

```
combined_pred = sign(direction_pred) × volatility_pred
```

This lets direction do what it's good at (54% accuracy) and
volatility do what it's good at (predicting move size).

In [11]:
@torch.no_grad()
def combined_predict(
    dir_model: nn.Module,
    vol_model: nn.Module,
    dir_loader: DataLoader,
    vol_loader: DataLoader,
    device: torch.device,
) -> dict:
    """
    Run both models and combine: sign(dir) × magnitude(vol).
    
    Returns dict with all predictions and targets for analysis.
    """
    dir_model.eval()
    vol_model.eval()
    
    # Direction predictions
    dir_preds, dir_targets = [], []
    for x, y in dir_loader:
        x = x.to(device)
        out = dir_model(x)
        preds = out[0] if isinstance(out, tuple) else out
        dir_preds.append(preds.cpu())
        dir_targets.append(y)
    dir_preds   = torch.cat(dir_preds)
    dir_targets = torch.cat(dir_targets)
    
    # Volatility predictions
    vol_preds, vol_targets = [], []
    for x, y in vol_loader:
        x = x.to(device)
        out = vol_model(x)
        preds = out[0] if isinstance(out, tuple) else out
        vol_preds.append(preds.cpu())
        vol_targets.append(y)
    vol_preds   = torch.cat(vol_preds)
    vol_targets = torch.cat(vol_targets)
    
    n = min(len(dir_preds), len(vol_preds))
    dir_preds   = dir_preds[:n]
    vol_preds   = vol_preds[:n]
    dir_targets = dir_targets[:n]
    vol_targets = vol_targets[:n]
    
    # Combined: sign from direction × magnitude from volatility
    combined = dir_preds.sign() * vol_preds
    
    # Metrics
    targets = dir_targets  # signed targets
    
    # Direction accuracy (from dir model alone)
    dir_acc = (dir_preds.sign() == targets.sign()).float().mean().item()
    
    # Volatility MAE (how well does vol model predict |return|?)
    vol_mae = (vol_preds - targets.abs()).abs().mean().item()
    
    # Baseline: always predict mean(|target|)
    naive_vol_mae = (targets.abs().mean() - targets.abs()).abs().mean().item()
    
    # Combined prediction correlation
    t, c = targets.numpy(), combined.numpy()
    corr = np.corrcoef(t, c)[0, 1] if t.std() > 0 and c.std() > 0 else 0.0
    
    # Direction-only correlation (sign * 1.0)
    dir_only_combined = dir_preds.sign().numpy()
    dir_corr = np.corrcoef(t, dir_only_combined)[0, 1] if t.std() > 0 else 0.0
    
    # Huber loss comparison
    huber = nn.HuberLoss(delta=1.0)
    combined_loss = huber(combined, targets).item()
    dir_only_loss = huber(dir_preds, targets).item()
    dummy_loss    = huber(torch.zeros_like(targets), targets).item()
    
    return {
        "dir_acc":          dir_acc,
        "vol_mae":          vol_mae,
        "naive_vol_mae":    naive_vol_mae,
        "combined_corr":    corr,
        "dir_only_corr":    dir_corr,
        "combined_loss":    combined_loss,
        "dir_only_loss":    dir_only_loss,
        "dummy_loss":       dummy_loss,
        "dir_preds":        dir_preds.numpy(),
        "vol_preds":        vol_preds.numpy(),
        "combined_preds":   c,
        "targets":          t,
    }

## Data & Feature Selection

In [12]:
train_df = pd.read_csv("data/train_features.csv", index_col=0, parse_dates=True)
val_df   = pd.read_csv("data/val_features.csv",   index_col=0, parse_dates=True)
test_df  = pd.read_csv("data/test_features.csv",  index_col=0, parse_dates=True)

cv_df   = pd.concat([train_df, val_df]).sort_index()
print(f"CV pool:  {len(cv_df):,} rows  |  "
      f"{cv_df.index.min().date()} → {cv_df.index.max().date()}")
print(f"Test set: {len(test_df):,} rows (held out)")

# ── Direction features (your working set) ──
dir_feature_cols = [
    "bull_regime",
    "z_bb_pct_b",
    "z_volume_z20",
    "z_log_close_return_1",
    "z_range",
    "z_ret_autocorr",
    "dow_cos",
    "high_vol_regime",
]

# ── Volatility features (focused on recent vol & volume) ──
# These predict HOW BIG the next move is, not which direction.
vol_feature_cols = [
    "z_vol_5",              # short-term realized vol
    "z_vol_20",             # medium-term realized vol
    "z_range",              # today's high-low range
    "z_atr_norm",           # normalized ATR
    "z_volume_z20",         # volume anomaly (big vol → big moves)
    "z_vol_ratio",          # short/long vol ratio (vol expansion)
    "high_vol_regime",      # binary regime flag
    "z_ret_autocorr",       # trending → larger moves
]

print(f"Direction features ({len(dir_feature_cols)}): {dir_feature_cols}")
print(f"Volatility features ({len(vol_feature_cols)}): {vol_feature_cols}")

CV pool:  14,574 rows  |  2015-04-03 → 2025-05-27
Test set: 1,710 rows (held out)
Direction features (8): ['bull_regime', 'z_bb_pct_b', 'z_volume_z20', 'z_log_close_return_1', 'z_range', 'z_ret_autocorr', 'dow_cos', 'high_vol_regime']
Volatility features (8): ['z_vol_5', 'z_vol_20', 'z_range', 'z_atr_norm', 'z_volume_z20', 'z_vol_ratio', 'high_vol_regime', 'z_ret_autocorr']


## Cross-Validation

In [13]:
from timeseries_cv import (
    expanding_window_folds, sliding_window_folds,
    apply_fold, CVFold,
)

@dataclass
class CombinedFoldResult:
    fold_idx:        int
    strategy:        str
    # Direction model
    dir_val_loss:    float
    dir_val_acc:     float
    # Volatility model
    vol_val_loss:    float
    vol_val_mae:     float
    naive_vol_mae:   float
    # Combined
    combined_corr:   float
    dir_only_corr:   float
    combined_loss:   float
    dir_only_loss:   float
    dummy_loss:      float
    # Meta
    n_train:         int
    n_val:           int
    train_start:     str
    train_end:       str
    val_start:       str
    val_end:         str


def cross_validate_combined(
    full_df:           pd.DataFrame,
    dir_feature_cols:  list[str],
    vol_feature_cols:  list[str],
    dir_model_factory,
    vol_model_factory,
    strategy:       Literal["expanding", "sliding"] = "expanding",
    n_folds:        int   = 5,
    train_frac:     float = 0.40,
    val_frac:       float = 0.10,
    gap_days:       int   = 0,
    seq_len:        int   = 20,
    batch_size:     int   = 64,
    # Direction training params
    dir_lr:           float = 1e-3,
    dir_criterion:    nn.Module | None = None,
    dir_weight_decay: float = 1e-2,
    # Volatility training params
    vol_lr:           float = 1e-3,
    vol_huber_delta:  float = 1.0,
    vol_weight_decay: float = 1e-3,
    # Shared
    epochs:         int   = 1000,
    patience:       int   = 20,
    warmup_epochs:  int   = 10,
    device:         torch.device = torch.device("cpu"),
    verbose:        bool  = True,
) -> list[CombinedFoldResult]:
    """Train direction + volatility models independently per fold, then combine."""
    
    if strategy == "expanding":
        folds = expanding_window_folds(
            full_df, n_folds=n_folds,
            min_train_frac=train_frac, val_frac=val_frac, gap_days=gap_days,
        )
    elif strategy == "sliding":
        folds = sliding_window_folds(
            full_df, n_folds=n_folds,
            train_frac=train_frac, val_frac=val_frac, gap_days=gap_days,
        )
    else:
        raise ValueError(f"Unknown strategy: {strategy!r}")
    
    if dir_criterion is None:
        dir_criterion = SoftDirectionalHuberLoss(delta=1.01)
    vol_criterion = nn.HuberLoss(delta=vol_huber_delta)
    
    results: list[CombinedFoldResult] = []
    
    for fold in folds:
        if verbose:
            print(f"\n{'='*70}")
            print(f"  [{fold.strategy.upper()}] Fold {fold.fold_idx}  |  "
                  f"train {fold.train_start.date()} → {fold.train_end.date()}  |  "
                  f"val {fold.val_start.date()} → {fold.val_end.date()}")
            print(f"{'='*70}")
        
        train_df, val_df = apply_fold(full_df, fold)
        
        # ── Direction model ────────────────────────────────────────
        if verbose:
            print(f"\n  ── Training DIRECTION model ──")
        
        dir_train_loader, dir_val_loader = make_loaders(
            train_df, val_df, dir_feature_cols, WindowDataset,
            seq_len=seq_len, batch_size=batch_size,
        )
        
        dir_model = dir_model_factory()
        dir_model = fit_model(
            dir_model, dir_train_loader, dir_val_loader,
            criterion=dir_criterion, device=device,
            lr=dir_lr, epochs=epochs, patience=patience,
            weight_decay=dir_weight_decay, warmup_epochs=warmup_epochs,
            tag="[DIR]", verbose=verbose,
        )
        dir_val = evaluate(dir_model, dir_val_loader, dir_criterion, device)
        
        # ── Volatility model ──────────────────────────────────────
        if verbose:
            print(f"\n  ── Training VOLATILITY model ──")
        
        vol_train_loader, vol_val_loader = make_loaders(
            train_df, val_df, vol_feature_cols, VolatilityDataset,
            seq_len=seq_len, batch_size=batch_size,
        )
        
        vol_model = vol_model_factory()
        vol_model = fit_model(
            vol_model, vol_train_loader, vol_val_loader,
            criterion=vol_criterion, device=device,
            lr=vol_lr, epochs=epochs, patience=patience,
            weight_decay=vol_weight_decay, warmup_epochs=warmup_epochs,
            tag="[VOL]", verbose=verbose,
        )
        vol_val = evaluate(vol_model, vol_val_loader, vol_criterion, device)
        
        # ── Combined evaluation ───────────────────────────────────
        combo = combined_predict(
            dir_model, vol_model,
            dir_val_loader, vol_val_loader,
            device=device,
        )
        
        if verbose:
            print(f"\n  ── FOLD {fold.fold_idx} RESULTS ──")
            print(f"  dir_acc       = {combo['dir_acc']:.3f}")
            print(f"  vol_mae       = {combo['vol_mae']:.3f}  (naive: {combo['naive_vol_mae']:.3f})")
            print(f"  combined_corr = {combo['combined_corr']:.3f}  (dir_only: {combo['dir_only_corr']:.3f})")
            print(f"  combined_loss = {combo['combined_loss']:.4f}  "
                  f"(dir_only: {combo['dir_only_loss']:.4f}  dummy: {combo['dummy_loss']:.4f})")
        
        results.append(CombinedFoldResult(
            fold_idx       = fold.fold_idx,
            strategy       = fold.strategy,
            dir_val_loss   = dir_val["loss"],
            dir_val_acc    = combo["dir_acc"],
            vol_val_loss   = vol_val["loss"],
            vol_val_mae    = combo["vol_mae"],
            naive_vol_mae  = combo["naive_vol_mae"],
            combined_corr  = combo["combined_corr"],
            dir_only_corr  = combo["dir_only_corr"],
            combined_loss  = combo["combined_loss"],
            dir_only_loss  = combo["dir_only_loss"],
            dummy_loss     = combo["dummy_loss"],
            n_train        = len(dir_train_loader.dataset),
            n_val          = len(dir_val_loader.dataset),
            train_start    = str(fold.train_start.date()),
            train_end      = str(fold.train_end.date()),
            val_start      = str(fold.val_start.date()),
            val_end        = str(fold.val_end.date()),
        ))
    
    return results


def summarize_combined_cv(results: list[CombinedFoldResult]) -> pd.DataFrame:
    if not results:
        print("No results.")
        return pd.DataFrame()
    
    strategy = results[0].strategy
    df = pd.DataFrame([vars(r) for r in results])
    
    print(f"\n{'='*70}")
    print(f"  {strategy.upper()}-WINDOW COMBINED CV SUMMARY")
    print(f"{'='*70}")
    for r in results:
        print(f"  Fold {r.fold_idx}  |  "
              f"dir_acc={r.dir_val_acc:.3f}  "
              f"vol_mae={r.vol_val_mae:.3f}  "
              f"combined_corr={r.combined_corr:.3f}  |  "
              f"loss: combined={r.combined_loss:.4f}  dir_only={r.dir_only_loss:.4f}  "
              f"dummy={r.dummy_loss:.4f}")
    
    print(f"\n  dir_acc        : {df['dir_val_acc'].mean():.3f} ± {df['dir_val_acc'].std():.3f}")
    print(f"  vol_mae        : {df['vol_val_mae'].mean():.3f} ± {df['vol_val_mae'].std():.3f}  "
          f"(naive: {df['naive_vol_mae'].mean():.3f})")
    print(f"  combined_corr  : {df['combined_corr'].mean():.3f} ± {df['combined_corr'].std():.3f}  "
          f"(dir_only: {df['dir_only_corr'].mean():.3f})")
    print(f"  combined_loss  : {df['combined_loss'].mean():.4f} ± {df['combined_loss'].std():.4f}")
    print(f"  dir_only_loss  : {df['dir_only_loss'].mean():.4f}")
    print(f"  dummy_loss     : {df['dummy_loss'].mean():.4f}")
    
    delta = df['dir_only_loss'].mean() - df['combined_loss'].mean()
    if delta > 0:
        print(f"\n  → Combined beats dir_only by {delta:.4f} on Huber loss ✓")
    else:
        print(f"\n  → Dir_only still beats combined by {-delta:.4f} — vol model hurting")
    
    print(f"{'='*70}")
    return df

## Model Config & Run

In [14]:
SEQ_LEN = 20

# ── Direction model factory (your working config) ──
def dir_model_factory():
    torch.manual_seed(SEED)
    return GRUWithAttention(
        input_size=len(dir_feature_cols),
        hidden_size=16,
        num_layers=2,
        linear_hidden=16,
        dropout=0.3,
    ).to(DEVICE)

# ── Volatility model factory (smaller, simpler) ──
def vol_model_factory():
    torch.manual_seed(SEED)
    return VolatilityGRU(
        input_size=len(vol_feature_cols),
        hidden_size=16,
        num_layers=1,         # single layer — vol clustering is a simple pattern
        linear_hidden=16,
        dropout=0.1,          # light reg — let it predict magnitude freely
    ).to(DEVICE)

shared_params = dict(
    full_df=cv_df,
    dir_feature_cols=dir_feature_cols,
    vol_feature_cols=vol_feature_cols,
    dir_model_factory=dir_model_factory,
    vol_model_factory=vol_model_factory,
    n_folds=5, val_frac=0.10,
    seq_len=SEQ_LEN, batch_size=64,
    dir_lr=1e-3, dir_weight_decay=2e-2,
    vol_lr=1e-3, vol_huber_delta=1.0, vol_weight_decay=1e-3,
    epochs=1000, patience=20, warmup_epochs=10,
    device=DEVICE,
)

In [15]:
exp_results = cross_validate_combined(**shared_params, strategy="expanding")
exp_df = summarize_combined_cv(exp_results)


  [EXPANDING] Fold 0  |  train 2015-04-03 → 2019-04-24  |  val 2019-04-25 → 2020-04-28

  ── Training DIRECTION model ──
  [DIR] Epoch   1 | train_loss=0.4478  val_loss=0.4188  val_mae=0.758  dir_acc=0.499
  [DIR] Epoch   2 | train_loss=0.4303  val_loss=0.4018  val_mae=0.736  dir_acc=0.507
  [DIR] Epoch   3 | train_loss=0.3820  val_loss=0.3922  val_mae=0.724  dir_acc=0.499
  [DIR] Epoch   4 | train_loss=0.3847  val_loss=0.3885  val_mae=0.718  dir_acc=0.503
  [DIR] Epoch   5 | train_loss=0.3714  val_loss=0.3888  val_mae=0.719  dir_acc=0.505
  [DIR] Epoch   6 | train_loss=0.3794  val_loss=0.3849  val_mae=0.714  dir_acc=0.506
  [DIR] Epoch   7 | train_loss=0.3621  val_loss=0.3838  val_mae=0.713  dir_acc=0.505
  [DIR] Epoch   8 | train_loss=0.3586  val_loss=0.3841  val_mae=0.715  dir_acc=0.506
  [DIR] Epoch   9 | train_loss=0.3525  val_loss=0.3815  val_mae=0.712  dir_acc=0.519
  [DIR] Epoch  10 | train_loss=0.3545  val_loss=0.3817  val_mae=0.710  dir_acc=0.518
  [DIR] Epoch  11 | train_lo

In [16]:
# sli_results = cross_validate_combined(**shared_params, strategy="sliding")
# sli_df = summarize_combined_cv(sli_results)

## Visualization

In [17]:
fold_labels = [f"Fold {r.fold_idx}" for r in exp_results]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=fold_labels,
    y=[r.dir_val_acc for r in exp_results],
    name="Dir. Accuracy", marker_color="#636efa", opacity=0.8,
))
fig.add_trace(go.Bar(
    x=fold_labels,
    y=[r.combined_corr for r in exp_results],
    name="Combined Corr", marker_color="#00cc96", opacity=0.8,
))
fig.add_trace(go.Bar(
    x=fold_labels,
    y=[r.dir_only_corr for r in exp_results],
    name="Dir-Only Corr", marker_color="#ef553b", opacity=0.5,
))
fig.add_hline(y=0.50, line_dash="dash", line_color="white",
              annotation_text="coin flip")
fig.update_layout(
    template="plotly_dark",
    title="Combined Model — Direction + Volatility",
    yaxis_title="Metric",
    yaxis_range=[-0.05, 0.70],
    barmode="group",
)
fig.show()

In [18]:
# Volatility model: does it beat naive?
fig = go.Figure()
fig.add_trace(go.Bar(
    x=fold_labels,
    y=[r.vol_val_mae for r in exp_results],
    name="Vol Model MAE", marker_color="#636efa", opacity=0.8,
))
fig.add_trace(go.Bar(
    x=fold_labels,
    y=[r.naive_vol_mae for r in exp_results],
    name="Naive MAE (predict mean)", marker_color="#ef553b", opacity=0.5,
))
fig.update_layout(
    template="plotly_dark",
    title="Volatility Model: Predicted |return| MAE vs Naive Baseline",
    yaxis_title="MAE",
    barmode="group",
)
fig.show()

In [19]:
# Loss comparison: combined vs direction-only vs dummy
fig = go.Figure()
fig.add_trace(go.Bar(
    x=fold_labels,
    y=[r.combined_loss for r in exp_results],
    name="Combined (dir+vol)", marker_color="#00cc96", opacity=0.8,
))
fig.add_trace(go.Bar(
    x=fold_labels,
    y=[r.dir_only_loss for r in exp_results],
    name="Dir-Only", marker_color="#636efa", opacity=0.8,
))
fig.add_trace(go.Bar(
    x=fold_labels,
    y=[r.dummy_loss for r in exp_results],
    name="Dummy (predict 0)", marker_color="#ef553b", opacity=0.5,
))
fig.update_layout(
    template="plotly_dark",
    title="Huber Loss: Combined vs Dir-Only vs Dummy",
    yaxis_title="Huber Loss",
    barmode="group",
)
fig.show()

## Holdout Evaluation

In [20]:
def evaluate_holdout_combined(
    train_df:          pd.DataFrame,
    eval_df:           pd.DataFrame,
    dir_feature_cols:  list[str],
    vol_feature_cols:  list[str],
    dir_model_factory,
    vol_model_factory,
    seq_len:       int   = 20,
    batch_size:    int   = 64,
    dir_lr:        float = 1e-3,
    dir_criterion: nn.Module | None = None,
    dir_weight_decay: float = 2e-2,
    vol_lr:        float = 1e-3,
    vol_huber_delta: float = 1.0,
    vol_weight_decay: float = 1e-3,
    epochs:        int   = 1000,
    patience:      int   = 20,
    warmup_epochs: int   = 10,
    device:        torch.device = torch.device("cpu"),
    ticker:        list[str] | None = None,
    verbose:       bool  = True,
):
    """Train both models on train_df, evaluate combined on eval_df."""
    
    if ticker is not None:
        eval_df = eval_df[eval_df["ticker"].isin(ticker)]
    
    # Split train into train/internal_val for early stopping
    dates = pd.DatetimeIndex(train_df.index).sort_values().unique()
    split_date = dates[int(len(dates) * 0.9)]
    internal_train = train_df.loc[train_df.index <= split_date]
    internal_val   = train_df.loc[train_df.index > split_date]
    
    tickers_str = ", ".join(ticker) if ticker else "all"
    if verbose:
        print(f"\n{'='*70}")
        print(f"  HOLDOUT EVALUATION  |  ticker={tickers_str}")
        print(f"  train {train_df.index.min().date()} -> {train_df.index.max().date()}")
        print(f"  eval  {eval_df.index.min().date()} -> {eval_df.index.max().date()}")
        print(f"{'='*70}")
    
    if dir_criterion is None:
        dir_criterion = SoftDirectionalHuberLoss(delta=1.01)
    vol_criterion = nn.HuberLoss(delta=vol_huber_delta)
    
    # ── Train direction model ──
    if verbose:
        print(f"\n  ── Training DIRECTION model ──")
    dir_train_l, dir_val_l = make_loaders(
        internal_train, internal_val, dir_feature_cols, WindowDataset,
        seq_len=seq_len, batch_size=batch_size,
    )
    dir_eval_l = DataLoader(
        WindowDataset(eval_df, feature_cols=dir_feature_cols, seq_len=seq_len),
        batch_size=batch_size, shuffle=False, num_workers=0,
    )
    
    dir_model = dir_model_factory()
    dir_model = fit_model(
        dir_model, dir_train_l, dir_val_l,
        criterion=dir_criterion, device=device,
        lr=dir_lr, epochs=epochs, patience=patience,
        weight_decay=dir_weight_decay, warmup_epochs=warmup_epochs,
        tag="[DIR]", verbose=verbose,
    )
    
    # ── Train volatility model ──
    if verbose:
        print(f"\n  ── Training VOLATILITY model ──")
    vol_train_l, vol_val_l = make_loaders(
        internal_train, internal_val, vol_feature_cols, VolatilityDataset,
        seq_len=seq_len, batch_size=batch_size,
    )
    vol_eval_l = DataLoader(
        VolatilityDataset(eval_df, feature_cols=vol_feature_cols, seq_len=seq_len),
        batch_size=batch_size, shuffle=False, num_workers=0,
    )
    
    vol_model = vol_model_factory()
    vol_model = fit_model(
        vol_model, vol_train_l, vol_val_l,
        criterion=vol_criterion, device=device,
        lr=vol_lr, epochs=epochs, patience=patience,
        weight_decay=vol_weight_decay, warmup_epochs=warmup_epochs,
        tag="[VOL]", verbose=verbose,
    )
    
    # ── Combined evaluation ──
    combo = combined_predict(
        dir_model, vol_model,
        dir_eval_l, vol_eval_l,
        device=device,
    )
    
    # Get eval dates
    eval_dates = []
    for tk, grp in eval_df.groupby("ticker"):
        grp = grp.sort_index()
        eval_dates.extend(grp.index[seq_len:].tolist())
    eval_dates = eval_dates[:len(combo["targets"])]
    
    if verbose:
        print(f"\n{'='*70}")
        print(f"  HOLDOUT RESULT  |  ticker={tickers_str}")
        print(f"{'='*70}")
        print(f"  dir_acc        : {combo['dir_acc']:.3f}")
        print(f"  vol_mae        : {combo['vol_mae']:.3f}  (naive: {combo['naive_vol_mae']:.3f})")
        print(f"  combined_corr  : {combo['combined_corr']:.3f}  (dir_only: {combo['dir_only_corr']:.3f})")
        print(f"  combined_loss  : {combo['combined_loss']:.4f}  "
              f"(dir_only: {combo['dir_only_loss']:.4f}  dummy: {combo['dummy_loss']:.4f})")
        print(f"{'='*70}")
    
    combo["eval_dates"] = eval_dates
    return combo

In [21]:
result = evaluate_holdout_combined(
    train_df=cv_df, eval_df=test_df,
    dir_feature_cols=dir_feature_cols,
    vol_feature_cols=vol_feature_cols,
    dir_model_factory=dir_model_factory,
    vol_model_factory=vol_model_factory,
    dir_lr=1e-3, vol_lr=1e-3, device=DEVICE,
    ticker=["BTC-USD", "ETH-USD"],
)


  HOLDOUT EVALUATION  |  ticker=BTC-USD, ETH-USD
  train 2015-04-03 -> 2025-05-27
  eval  2025-05-28 -> 2026-03-08

  ── Training DIRECTION model ──
  [DIR] Epoch   1 | train_loss=0.4286  val_loss=0.4096  val_mae=0.754  dir_acc=0.491
  [DIR] Epoch   2 | train_loss=0.3910  val_loss=0.3929  val_mae=0.740  dir_acc=0.512
  [DIR] Epoch   3 | train_loss=0.3826  val_loss=0.3863  val_mae=0.738  dir_acc=0.524
  [DIR] Epoch   4 | train_loss=0.3726  val_loss=0.3827  val_mae=0.737  dir_acc=0.532
  [DIR] Epoch   5 | train_loss=0.3670  val_loss=0.3783  val_mae=0.735  dir_acc=0.540
  [DIR] Epoch   6 | train_loss=0.3643  val_loss=0.3772  val_mae=0.737  dir_acc=0.538
  [DIR] Epoch   7 | train_loss=0.3632  val_loss=0.3782  val_mae=0.735  dir_acc=0.535
  [DIR] Epoch   8 | train_loss=0.3588  val_loss=0.3807  val_mae=0.736  dir_acc=0.530
  [DIR] Epoch   9 | train_loss=0.3627  val_loss=0.3813  val_mae=0.736  dir_acc=0.533
  [DIR] Epoch  10 | train_loss=0.3623  val_loss=0.3769  val_mae=0.733  dir_acc=0.540


In [22]:
# Combined prediction vs actual
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=result["eval_dates"], y=result["targets"],
    name="actual", opacity=0.5,
))
fig.add_trace(go.Scatter(
    x=result["eval_dates"], y=result["combined_preds"],
    name="combined (dir × vol)", opacity=0.7,
))
fig.add_trace(go.Scatter(
    x=result["eval_dates"], y=result["dir_preds"],
    name="direction only", opacity=0.3,
))
fig.update_layout(template="plotly_dark", title="Holdout: Combined vs Direction-Only vs Actual")
fig.show()

In [23]:
# Volatility: predicted magnitude vs actual |return|
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=result["eval_dates"], y=np.abs(result["targets"]),
    name="|actual return|", opacity=0.4,
))
fig.add_trace(go.Scatter(
    x=result["eval_dates"], y=result["vol_preds"],
    name="predicted magnitude", opacity=0.7,
))
fig.update_layout(template="plotly_dark", title="Volatility Model: Predicted vs Actual |Return|")
fig.show()